In [29]:
from vistiq.segment import MicroSAMSegmenter, MicroSAMSegmenterConfig, SegmentationFlow, SegmentationFlowConfig
from vistiq.io import ImageLoader, ImageLoaderConfig
from vistiq.core import FuncProcessor, FuncProcessorConfig, Tiler, TilerConfig, Untiler, UntilerConfig
from vistiq.preprocess import ResizeConfig, Resize, RescaleConfig, Rescale, DoG, DoGConfig, PreprocessorConfig, Preprocessor
from vistiq.segment import RegionFilterConfig, RegionFilter, RangeFilterConfig, RangeFilter, RegionAnalyzerConfig, RegionAnalyzer 
from vistiq.utils import ArrayIteratorConfig 
from vistiq.analysis import CoincidenceDetectorConfig, CoincidenceDetector
import supervision as sv

from skimage.filters import gaussian
from skimage.exposure import rescale_intensity, adjust_sigmoid, adjust_gamma
import stackview
import os
import numpy as np
from joblib import Parallel, delayed
import math
import logging

In [30]:
import vistiq
logger = logging.getLogger(vistiq.__name__)

import torch
logger.info(f"Torch version: {torch.__version__}. Cuda enabled: {torch.cuda.is_available()}")

2026-05-23 23:47:45,173 - INFO - Torch version: 2.11.0+cu126. Cuda enabled: True


# Functions and Classes

In [31]:
from typing import Literal

In [32]:
def box_iou_batch_3d(
    boxes_true: np.typing.NDArray[np.number],
    boxes_detection: np.typing.NDArray[np.number],
    overlap_metric: Literal["IOU", "IOS"] = "IOU"
) -> np.ndarray[np.float32]:
    """
    Adapted for 3d from https://github.com/roboflow/supervision/blob/develop/src/supervision/detection/utils/iou_and_nms.py
    
    Compute pairwise overlap scores between batches of bounding boxes.

    Supports standard IOU (intersection-over-union) and IOS
    (intersection-over-smaller-area) metrics for all `boxes_true` and
    `boxes_detection` pairs. Returns a matrix of overlap values in range
    `[0, 1]`, matching each box from the first batch to each from the second.

    Args:
        boxes_true: Array of reference boxes in
            shape `(N, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_max)`.
        boxes_detection: Array of detected boxes in
            shape `(M, 4)` as `(x_min, y_min, z_min, x_max, y_max, z_min)`.
        overlap_metric: Overlap type.
            Use `OverlapMetric.IOU` for intersection-over-union,
            `OverlapMetric.IOS` for intersection-over-smaller-area.
            Defaults to `OverlapMetric.IOU`.

    Returns:
        Overlap matrix of shape `(N, M)`, where entry
            `[i, j]` is the overlap between `boxes_true[i]` and
            `boxes_detection[j]`.

    Raises:
        ValueError: If `overlap_metric` is not IOU or IOS.

    Examples:
        ```pycon
        >>> import numpy as np
        >>> import supervision as sv
        >>> boxes_true = np.array([
        ...     [100, 100, 200, 200],
        ...     [300, 300, 400, 400]
        ... ])
        >>> boxes_detection = np.array([
        ...     [150, 150, 250, 250],
        ...     [320, 320, 420, 420]
        ... ])
        >>> sv.box_iou_batch_3d(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOU
        ... )
        array([[0.14285..., 0.        ],
               [0.        , 0.47058...]], dtype=float32)
        >>> sv.box_iou_batch(
        ...     boxes_true, boxes_detection, overlap_metric=sv.OverlapMetric.IOS
        ... )
        array([[0.25, 0.  ],
               [0.  , 0.64]], dtype=float32)

        ```
    """
    #overlap_metric = OverlapMetric.from_value(overlap_metric)
    x_min_true, y_min_true, z_min_true, x_max_true, y_max_true, z_max_true = boxes_true.T
    x_min_det, y_min_det, z_min_det, x_max_det, y_max_det, z_max_det = boxes_detection.T
    count_true, count_det = boxes_true.shape[0], boxes_detection.shape[0]

    if count_true == 0 or count_det == 0:
        return cast(
            np.typing.NDArray[np.float32], np.empty((count_true, count_det), dtype=np.float32)
        )

    x_min_inter = np.empty((count_true, count_det), dtype=np.float32)
    x_max_inter = np.empty_like(x_min_inter)
    y_min_inter = np.empty_like(x_min_inter)
    y_max_inter = np.empty_like(x_min_inter)
    z_min_inter = np.empty_like(x_min_inter)
    z_max_inter = np.empty_like(x_min_inter)

    np.maximum(x_min_true[:, None], x_min_det[None, :], out=x_min_inter)
    np.minimum(x_max_true[:, None], x_max_det[None, :], out=x_max_inter)
    np.maximum(y_min_true[:, None], y_min_det[None, :], out=y_min_inter)
    np.minimum(y_max_true[:, None], y_max_det[None, :], out=y_max_inter)
    np.maximum(z_min_true[:, None], z_min_det[None, :], out=z_min_inter)
    np.minimum(z_max_true[:, None], z_max_det[None, :], out=z_max_inter)

    # we reuse x_max_inter and y_max_inter to store inter_w, inter_h and inter_d
    np.subtract(x_max_inter, x_min_inter, out=x_max_inter)  # inter_w
    np.subtract(y_max_inter, y_min_inter, out=y_max_inter)  # inter_h
    np.subtract(z_max_inter, z_min_inter, out=z_max_inter)  # inter_d
    np.clip(x_max_inter, 0.0, None, out=x_max_inter)
    np.clip(y_max_inter, 0.0, None, out=y_max_inter)
    np.clip(z_max_inter, 0.0, None, out=z_max_inter)

    area_inter = x_max_inter * y_max_inter * z_max_inter # inter_w * inter_h * inter_d

    area_true = (x_max_true - x_min_true) * (y_max_true - y_min_true) * (z_max_true - z_min_true)
    area_det = (x_max_det - x_min_det) * (y_max_det - y_min_det)  * (z_max_det - z_min_det)

    if overlap_metric == "IOU":
        area_norm = area_true[:, None] + area_det[None, :] - area_inter
    elif overlap_metric == "IOS":
        area_norm = np.minimum(area_true[:, None], area_det[None, :])
    else:
        raise ValueError(
            f"overlap_metric {overlap_metric} is not supported, "
            "only 'IOU' and 'IOS' are supported"
        )

    out: np.ndarray[np.float32] = np.zeros_like(area_inter, dtype=np.float32)
    np.divide(area_inter, area_norm, out=out, where=area_norm > 0)
    return out

In [33]:
def ac_3d(img, init=None, evolve_init=False, dtype="auto", out_max=1, points=100, margin=5, **kwargs):
    if img.ndim == 2:
        img = np.expand_dims(img, axis=0) 

    shape_2d = img.shape[-2:]
    if init is None:
        logger.info(f"creating initial rectangular snake with {points} points and a margin of {margin}")
        r, c = shape_2d
        init = np.array([
            np.concatenate([np.linspace(margin, c-2*margin, points), np.full(points, c-margin), 
                            np.linspace(c-2*margin, margin, points), np.full(points, margin)]),
            np.concatenate([np.full(points, margin), np.linspace(margin, r-2*margin, points), 
                            np.full(points, r-margin), np.linspace(r-2*margin, margin, points)])
        ]).T
    logger.debug(f"img.dtype={img.dtype}, img.shape={img.shape}, init.shape={init.shape}")
    if dtype == "auto":
        dtype = img.dtype
    if evolve_init:
        isnake_mask = np.zeros(img.shape, dtype=dtype)
        isnake_points = np.zeros((img.shape[0], init.shape[0], init.shape[1],), dtype="float64")
        for z in reversed(range(img.shape[0])):
            isnake = segmentation.active_contour(img[z], init, **kwargs)
            isnake_mask[z] = draw.polygon2mask(shape_2d, isnake).astype(dtype)*out_max
            init = isnake
            isnake_points[z] = isnake
    else:
        isnake_points = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(segmentation.active_contour)(i, init, **kwargs) for i in img))
        isnake_mask = np.array(Parallel(n_jobs=-1, verbose=10)(delayed(draw.polygon2mask)(shape_2d, isnake) for isnake in isnake_points))
        isnake_mask = isnake_mask.astype(dtype)*out_max
    return isnake_mask.squeeze(), isnake_points.squeeze()


In [34]:
def labels_to_masks(labels):
    label_values = (v for v in np.unique(labels) if v > 0)
    masks = []
    for value in label_values:
        mask = labels == value
        masks.append(mask)
    return np.array(masks)

In [35]:
def group_bboxes(bboxes, divisor=1, threshold=0.5):
    
    def in_groups(item, groups):
        for g in groups:
            if item in g:
                return True
        return False
    
    xyxy = np.mod(bboxes, divisor)
    #print (xyxy[:7])
    if len(bboxes[0]) == 4:
        iou_matrix = sv.box_iou_batch(xyxy, xyxy, overlap_metric=sv.OverlapMetric.IOU)
    elif len(bboxes[0]) == 6:
        iou_matrix = box_iou_batch_3d(xyxy, xyxy, overlap_metric="IOU")
    #print (iou_matrix)
    iou_matrix = np.triu(iou_matrix, k=1)
    pairs = np.argwhere(iou_matrix > threshold)
    
    groups = []
    for i, pair in enumerate(pairs):
        p0 = pair[0]
        #print (i, p0, p1, iou_matrix[p0, p1])
        if not in_groups(p0, groups):
            pairs_with_p0 = np.unique(np.array([p for p in pairs if p[0] == p0]).flatten())
            logger.info(f"Creating new group with {pairs_with_p0}")
            groups.append(pairs_with_p0)
    return groups

In [36]:
def label_grouped_mask(mask, groups:list[np.ndarray], threshold=1):
    labels = []
    th = math.prod(tile_factor)//2
    for label_value, g in enumerate(groups, 1):
        label_array = (mask[g].sum(axis=0)>threshold)*label_value
        labels.append(label_array)
    labels = np.sum(np.array(labels), axis=0).astype("uint16")
    return labels

# Load Image

In [37]:
#path="/standard/vol191/siegristlab/Microsam_Segmentation/Conditional Split/control_24+48/DCP1/1_Dpn.tif"
path="/standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif"

scene_index = 0

embedding_path = "/standard/vol191/siegristlab/Sagar/microsam/embeddings/"

In [38]:
ilc = ImageLoaderConfig(
    squeeze=True, 
    rename_channel={"Red": "Dpn", "Green": "Scrib", "Blue": "EdU"}, 
    scene_index=scene_index, 
    split_channels=True, 
    #substack="C:1"
)
img, metadata = ImageLoader(ilc).run(path)
metadata

2026-05-23 23:47:47,409 - INFO - Loading image from: /standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif
2026-05-23 23:47:47,494 - INFO - Scenes found: ('Series001', 'Series002', 'Series003')
2026-05-23 23:47:47,653 - INFO - Loaded image: /standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1.lif scene=0 -> shape=(3, 93, 512, 512) dtype=uint8, channel_names=['Scrib', 'EdU', 'Dpn']
2026-05-23 23:47:47,655 - INFO - Loaded image with shape: (3, 93, 512, 512), dtype: uint8
2026-05-23 23:47:47,656 - INFO - Finished in state Completed()


{'scene_index': 0,
 'dim_order': 'CZYX',
 'axes': ['C', 'Z', 'Y', 'X'],
 'channel_names': ['Scrib', 'EdU', 'Dpn'],
 'channel_axis': 0,
 'shape': (3, 93, 512, 512),
 'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
 'pixel_unit': 'um',
 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
 'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)}

# Preprocess

In [39]:
# Rescale intensity for each channel
scfg = RescaleConfig(
    low=2, 
    high=98, 
    dtype=np.uint8, 
    iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
)
simg,_ = Rescale(scfg).run(img, metadata=metadata, verbose=1)

2026-05-23 23:47:47,938 - INFO - Running preprocessor Rescale, on stack of type uint8, True
2026-05-23 23:47:48,072 - INFO - Running Rescale with config: classname='RescaleConfig' package='vistiq.preprocess.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=<class 'numpy.uint8'> low=2.0 high=98.0
2026-05-23 23:47:48,072 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:48,072 - INFO - Using Parallel with n_jobs=-1 for 3 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   3 out of   3 | elapsed:    0.5s finished
2026-05-23 23:47:48,559 - INFO - Reshaped results 

In [40]:
# Apply gaussian blur, separately for each channel and focal plane
gcfg = FuncProcessorConfig(
    func=gaussian,
    kwargs={"sigma": 1.0},
    iterator_config=ArrayIteratorConfig(slice_def=(-2,-1))
)
nimg, _ = FuncProcessor(gcfg).run(simg, verbose=1)    

2026-05-23 23:47:49,363 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=True dtype=None func=<function gaussian at 0x7f241e732fc0> args=[] kwargs={'sigma': 1.0}
2026-05-23 23:47:49,363 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:49,364 - INFO - Using Parallel with n_jobs=-1 for 279 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done 257 tasks      | elapsed:    0.2s
[Parallel(n_jobs=-1)]: Done 279 out of 279 | elapsed:    0.3s finished
2026-05-23 23:47:49,797 - INFO - Reshaped result

In [41]:
gamma = 0.2
enimg = np.array([rescale_intensity(adjust_sigmoid(adjust_gamma(i, gamma=gamma)),out_range="uint8") for i in nimg])
print (np.max(enimg))
#emimg = exposure.rescale_intensity(filters.gaussian(exposure.adjust_sigmoid(exposure.adjust_gamma(mimg, gamma=gamma)), sigma=5.0), out_range="uint8")

255


In [42]:
stackview.slice(np.concatenate([simg, rescale_intensity(nimg, out_range="uint8"), enimg], axis=-1))

In [43]:
# Project all channels to one.

pcfg = FuncProcessorConfig(
    func=np.max, 
    kwargs={"axis":("C")}, 
    strict_axis=False,
    dtype=np.uint16,
)
c_img, c_metadata = FuncProcessor(pcfg).run(enimg, metadata=metadata)
metadata, c_metadata

2026-05-23 23:47:53,619 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=False dtype=<class 'numpy.uint16'> func=<function max at 0x7f243c452c70> args=[] kwargs={'axis': 'C'}
2026-05-23 23:47:53,619 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:53,619 - INFO - Mapped axis letters C to axis indices (0,)
2026-05-23 23:47:53,640 - INFO - Converted results to uint16
2026-05-23 23:47:53,640 - INFO - Dropping axes. New axes: ['Z', 'Y', 'X']
2026-05-23 23:47:53,640 - INFO - Updating metadata with new shape ratio: [1. 1. 1.]
2026-05-23 23:47:53,641 - INFO - Metadata 

({'scene_index': 0,
  'dim_order': 'CZYX',
  'axes': ['C', 'Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (3, 93, 512, 512),
  'dims': <Dimensions [C: 3, Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)})

In [44]:
stackview.slice(c_img, continuous_update=True)

# Rough Mask of Projection to inform Lobe Segmentation

In [45]:
#c_mask, c_points = ac_3d(proj, points=100, margin=0, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") # beta=0.1, w_edge=1.75
#logging.info(f"{proj.dtype}, {c_mask.dtype}, {np.max(proj)}, {np.max(c_mask)}")

In [46]:
#stackview.slice(np.concatenate([proj, animg[40], c_mask, (proj-0.2*c_mask)], axis=-1))

In [47]:
#isnake_img,_ = ac_3d(animg[::4], init=c_points, out_max=255, alpha=0.015, beta=0.1, gamma=0.001, w_edge=1.75, boundary_condition="periodic") #w_edge=1.75

In [48]:
#stackview.slice(np.concatenate([img[0,::4]+0.2*isnake_img, animg[::4]+0.2*isnake_img, isnake_img], axis=-1))

# Detect Tissue boundaries with MicroSAM (resampling)

## Resize

In [49]:
factor = 3
width = c_img.shape[-1]//factor
padding = 10

rcfg = ResizeConfig(width=width)
r_img, r_metadata = Resize(rcfg).run(c_img, metadata=c_metadata, verbose=1)
c_metadata, r_metadata

2026-05-23 23:47:53,785 - INFO - Resizing stack from (93, 512, 512) to [93, 170, 170]
2026-05-23 23:47:53,876 - INFO - Running preprocessor Resize, on stack of type uint16, True
2026-05-23 23:47:53,966 - INFO - Running Resize with config: classname='ResizeConfig' package='vistiq.preprocess.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=(93, 170, 170) output_axes=None recompute_scale=True squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=None width=170 height=None order=1 preserve_range=True anti_aliasing=True
2026-05-23 23:47:53,966 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:53,967 - INFO - Using Parallel with n_jobs=-1 for 93 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent worker

({'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 512, 512),
  'dims': <Dimensions [Z: 93, Y: 512, X: 512]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)},
 {'scene_index': 0,
  'dim_order': 'ZYX',
  'axes': ['Z', 'Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (93, 170, 170),
  'dims': <Dimensions [Z: 93, Y: 170, X: 170]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)})

In [50]:
stackview.orthogonal(r_img)

## Z-Project

In [51]:
fcfg = FuncProcessorConfig(
    func=np.mean, 
    kwargs={"axis":("Z")},
)
proj, p_metadata = FuncProcessor(fcfg).run(r_img, metadata=r_metadata)

2026-05-23 23:47:54,225 - INFO - Running FuncProcessor with config: classname='FuncProcessorConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None strict_axis=True dtype=None func=<function mean at 0x7f243c453cf0> args=[] kwargs={'axis': 'Z'}
2026-05-23 23:47:54,225 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:54,225 - INFO - Mapped axis letters Z to axis indices (0,)
2026-05-23 23:47:54,227 - INFO - Dropping axes. New axes: ['Y', 'X']
2026-05-23 23:47:54,227 - INFO - Updating metadata with new shape ratio: [1. 1.]
2026-05-23 23:47:54,228 - INFO - Metadata updated in FuncProcessor: 4 key(s) changed
2026-05-23 23:47:54,228 - INFO -   dims: <Di

In [52]:
tile_factor = (factor, factor)
tcfg = TilerConfig(factor=tile_factor, alt_flip=False, pad_width={-2:(0,padding),-1:(0,padding)})
t_proj,t_metadata = Tiler(tcfg).run(proj, metadata=p_metadata)

2026-05-23 23:47:54,247 - INFO - Running Tiler with config: classname='TilerConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3) pad_width={-2: (0, 10), -1: (0, 10)} pad_kwargs={'mode': 'constant', 'constant_values': 0} alt_flip=False
2026-05-23 23:47:54,248 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:47:54,249 - INFO - Updating metadata with new shape ratio: [0.33333333 0.33333333]
2026-05-23 23:47:54,249 - INFO - Metadata updated in Tiler: 2 key(s) changed
2026-05-23 23:47:54,249 - INFO -   dims: <Dimensions [Y: 170, X: 170]> -> <Dimensions [Y: 540, X: 540]>
2026-05-23 23:47:54,250 - INFO -   shape: (170, 170) -> (540, 540)
20

({'scene_index': 0,
  'dim_order': 'YX',
  'axes': ['Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (170, 170),
  'dims': <Dimensions [Y: 170, X: 170]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)},
 {'scene_index': 0,
  'dim_order': 'YX',
  'axes': ['Y', 'X'],
  'channel_names': ['Scrib', 'EdU', 'Dpn'],
  'channel_axis': 0,
  'shape': (540, 540),
  'dims': <Dimensions [Y: 540, X: 540]>,
  'pixel_unit': 'um',
  'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506),
  'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.9038105490963506, X=0.9038105490963506)})

In [53]:
rcfg = RegionFilterConfig(filters=[
    RangeFilter(
        RangeFilterConfig(
            attribute="circularity", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="aspect_ratio", range=(0.5,1.0)
        ),
    ),
    RangeFilter(
        RangeFilterConfig(
            attribute="cross_sectional_area", range=(1500,np.inf)
        ),
    )]
)
rf = RegionFilter(rcfg)

mcfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mcfg)

scfg = SegmentationFlowConfig(
    segmenter = microsam,
    region_filter = rf,
)

p_labels = SegmentationFlow(scfg).run(t_proj, metadata=t_metadata)

2026-05-23 23:47:56,977 - INFO - RegionAnalyzer not provided, using default RegionAnalyzer with properties: ['label', 'centroid', 'circularity', 'aspect_ratio', 'cross_sectional_area']
2026-05-23 23:47:56,978 - INFO - Segmenter config: classname='SegmentationFlowConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None segmenter=MicroSAMSegmenter(classname='MicroSAMSegmenterConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' predictor=<segment_anything.predictor.SamPredictor object at 0x7f22001a3380> segmenter=<micro_sam.instance_segmentation.InstanceSegmentationWithDecoder object at 0x7f220015fb00> checkpoint

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-05-23 23:48:01,405 - INFO - Finished in state Completed()


In [54]:
racfg = RegionAnalyzerConfig(
    properties=["area", "cross_sectional_area", "bbox", "aspect_ratio", "circularity", "perimeter"],
    iterator_config = ArrayIteratorConfig(slice_def=()),
    output_type="dataframe"
)
ra = RegionAnalyzer(racfg)

p_results = ra.run(p_labels, metadata=t_metadata)

2026-05-23 23:48:01,427 - INFO - Running RegionAnalyzer with config: classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'area', 'cross_sectional_area', 'bbox', 'aspect_ratio', 'circularity', 'perimeter']
2026-05-23 23:48:01,428 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:48:01,433 - INFO - RegionAnalyzer: Applying scale: (0.9038105490963506, 0.9038105490963506)
2026-05-23 23:48:01,457 - INFO - Identified 18 regions, return as dataframe
2026-05-23 23:48:01,460 - INFO - Finished in state Completed()
2026-05-23 23:48:01,465 - INFO - Finished in state Completed()


In [55]:
p_results.describe()

,area,bbox-0,bbox-1,bbox-2,bbox-3,perimeter,circularity,aspect_ratio,cross_sectional_area
count,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000,18.000000
mean,5456.351983,222.500000,211.944444,311.222222,312.555556,282.135994,0.861437,0.751281,5456.351983
std,171.665403,157.703463,152.035395,155.827414,153.001047,6.045039,0.012396,0.021795,171.665403
min,5256.581028,0.000000,17.000000,93.000000,110.000000,275.203465,0.827520,0.723363,5256.581028
25%,5292.523463,83.500000,46.000000,168.250000,155.000000,276.536471,0.856832,0.730806,5292.523463
50%,5433.434143,223.000000,211.500000,312.000000,312.500000,281.110349,0.862452,0.749390,5433.434143
75%,5616.413809,360.750000,377.750000,454.500000,470.000000,287.133536,0.870456,0.765733,5616.413809
max,5724.649549,446.000000,407.000000,528.000000,515.000000,291.819820,0.875501,0.793927,5724.649549


In [56]:
stackview.blend(t_proj, p_labels, blend_factor=40, continuous_update=True)

# Consensus voting
Find matching regions based on IoU of bounding boxes, then create consensus masks (only consider pixels that show up in at least half of the masks).

In [57]:
logger.info(f"width={width}, p_labels.shape={p_labels.shape}, {p_labels.shape[-1]//factor}")
groups = group_bboxes(p_results[["bbox-1", "bbox-0", "bbox-3","bbox-2"]].to_numpy()-np.array((0,0,1,1)), divisor=p_labels.shape[-1]//factor, threshold=0.5)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)

masks = labels_to_masks(p_labels)
untiled,_ = Untiler(ucfg).run(masks)
#untiled = untile(masks, tile_factor)
u_proj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(u_proj, groups, threshold=math.prod(tile_factor)//2)
labels = labels[..., 0:labels.shape[-2]-padding, 0:labels.shape[-1]-padding]
stackview.blend(proj, labels, blend_factor=25, continuous_update=True)

2026-05-23 23:48:07,655 - INFO - width=170, p_labels.shape=(540, 540), 180
2026-05-23 23:48:07,666 - INFO - Creating new group with [ 0  1  2  6  7  8 12 13 14]
2026-05-23 23:48:07,667 - INFO - Creating new group with [ 3  4  5  9 10 11 15 16 17]
2026-05-23 23:48:07,694 - INFO - Running Untiler with config: classname='UntilerConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-05-23 23:48:07,694 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:48:07,696 - INFO - factor=(3, 3), array.shape=(18, 540, 540), vs.shape=(3, 18, 540, 180), hs.shape=(3, 3, 18, 180, 180), untiled.shape=(9, 18, 180, 180)
2026-05-23 23:48:07,698 - INFO - Fi

# Segment tiled Z-stack

In [58]:
tr_img,tr_metadata = Tiler(tcfg).run(r_img, metadata=r_metadata)

2026-05-23 23:48:09,227 - INFO - Running Tiler with config: classname='TilerConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3) pad_width={-2: (0, 10), -1: (0, 10)} pad_kwargs={'mode': 'constant', 'constant_values': 0} alt_flip=False
2026-05-23 23:48:09,228 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-23 23:48:09,301 - INFO - Updating metadata with new shape ratio: [1.         0.33333333 0.33333333]
2026-05-23 23:48:09,302 - INFO - Metadata updated in Tiler: 2 key(s) changed
2026-05-23 23:48:09,302 - INFO -   dims: <Dimensions [Z: 93, Y: 170, X: 170]> -> <Dimensions [Z: 93, Y: 540, X: 540]>
2026-05-23 23:48:09,303 - INFO -   shape: (9

In [95]:
mcfg = MicroSAMSegmenterConfig(
    iterator_config=ArrayIteratorConfig(slice_def=()),
    embedding_path=embedding_path,
)
microsam = MicroSAMSegmenter(mcfg)

scfg = SegmentationFlowConfig(
    segmenter = microsam,
    region_filter = RegionFilter(
        RegionFilterConfig(
            filters=[
                RangeFilter(
                    RangeFilterConfig(
                        attribute="cross_sectional_area", 
                        range=(3000, np.inf)
                    )
                ),
                RangeFilter(
                    RangeFilterConfig(
                        attribute="aspect_ratio", 
                        range=(0.5, 1.0)
                    )
                ),
            ]
        )
    )
)

tlabels = SegmentationFlow(scfg).run(tr_img, metadata=tr_metadata)

2026-05-24 00:20:19,143 - INFO - RegionAnalyzer not provided, using default RegionAnalyzer with properties: ['label', 'centroid', 'cross_sectional_area', 'aspect_ratio']
2026-05-24 00:20:19,144 - INFO - Segmenter config: classname='SegmentationFlowConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None segmenter=MicroSAMSegmenter(classname='MicroSAMSegmenterConfig' package='vistiq.segment.label' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None model_type='vit_l_lm' predictor=<segment_anything.predictor.SamPredictor object at 0x7f21cc8ef7a0> segmenter=<micro_sam.instance_segmentation.InstanceSegmentationWithDecoder object at 0x7f21cc8e46e0> checkpoint=None embedding

DEBUG: entered LabelRemover.run
DEBUG: region_properties type = <class 'list'>


2026-05-24 00:20:54,802 - INFO - Finished in state Completed()
2026-05-24 00:20:55,321 - INFO - Finished in state Completed()
2026-05-24 00:20:55,491 - INFO - HTTP Request: POST https://api.prefect.cloud/api/accounts/fe04c388-039f-4a9a-8a38-6ddd01791c77/workspaces/0aefefee-2869-4637-89ac-c510d86c639e/flow_runs/06a127c8-3596-7976-8000-821c0c812a23/set_state "HTTP/1.1 201 Created"
2026-05-24 00:20:55,984 - INFO - Finished in state Completed()


In [96]:
ra=RegionAnalyzer(
   RegionAnalyzerConfig(
        output_type="dataframe", 
        properties=["cross_sectional_area", "aspect_ratio", "bbox", "volume", "aspect_ratio"],
        iterator_config=ArrayIteratorConfig(slice_def=(-3,-2,-1))
    )
)
tresults = ra.run(tlabels, metadata=tr_metadata)

2026-05-24 00:20:56,396 - INFO - Running RegionAnalyzer with config: classname='RegionAnalyzerConfig' package='vistiq.segment.analysis' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-3, -2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='dataframe' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None properties=['label', 'cross_sectional_area', 'aspect_ratio', 'bbox', 'volume', 'aspect_ratio']
2026-05-24 00:20:56,396 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-24 00:20:56,841 - INFO - RegionAnalyzer: Applying scale: (-0.9999284782608696, 0.9038105490963506, 0.9038105490963506)
2026-05-24 00:20:57,298 - INFO - Identified 17 regions, return as dataframe
2026-05-24 00:20:57,300 - INFO - Finished in state Completed()
2026-05-24 00:20:57,305 - INFO - Finished in state Completed()

In [99]:
stackview.blend(tr_img, tlabels, blend_factor=25, continuous_update=True)

In [100]:
tresults.describe()

,bbox-0,bbox-1,bbox-2,bbox-3,bbox-4,bbox-5,aspect_ratio,cross_sectional_area,volume
count,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000,17.000000
mean,19.176471,220.411765,187.176471,90.882353,339.176471,313.470588,0.694454,6911.422603,310397.179520
std,5.592800,156.161398,149.313527,0.485071,155.922351,146.258383,0.028975,1928.442499,31666.466244
min,0.000000,0.000000,11.000000,89.000000,110.000000,112.000000,0.610490,5321.930909,282563.292607
25%,17.000000,70.000000,27.000000,91.000000,170.000000,156.000000,0.679083,5573.527950,296427.911850
50%,21.000000,203.000000,189.000000,91.000000,350.000000,328.000000,0.690913,5910.079835,304557.672386
75%,22.000000,360.000000,374.000000,91.000000,500.000000,471.000000,0.712651,8065.809024,309226.587408
max,24.000000,433.000000,392.000000,91.000000,530.000000,516.000000,0.740502,12754.662964,415707.418632


In [101]:
tgroups = group_bboxes(tresults[["bbox-2", "bbox-1", "bbox-0", "bbox-5", "bbox-4", "bbox-3"]].to_numpy()-np.array((0,0,0,1,1,1)), divisor=tlabels.shape[-1]//factor, threshold=0.5)

tmasks = labels_to_masks(tlabels)

ucfg = UntilerConfig(
    factor = tile_factor,
    iterator_config = ArrayIteratorConfig(slice_def=())
)
untiled,_ = Untiler(ucfg).run(tmasks)
tproj =  np.sum(untiled>0, axis=0)>0

labels = label_grouped_mask(tproj, tgroups, threshold=math.prod(tile_factor)//2)

# remove padding on bottom and right edge
cropped_height = labels.shape[-2]-padding
cropped_width = labels.shape[-1]-padding
cropped_labels = labels[..., 0:cropped_height, 0:cropped_width]

2026-05-24 00:22:56,598 - INFO - Creating new group with [ 0  5  7  9 11 12 13 16]
2026-05-24 00:22:56,599 - INFO - Creating new group with [ 1  3  4  6 10]
2026-05-24 00:22:56,600 - INFO - Creating new group with [ 2  3  4  6  8 10 15]


width=170, tlabels.shape=(93, 540, 540), 180


2026-05-24 00:22:58,131 - INFO - Running Untiler with config: classname='UntilerConfig' package='vistiq.core' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=()) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=None output_axes=None recompute_scale=False squeeze=True split_axis=None split_channels=False rename_channel=None factor=(3, 3)
2026-05-24 00:22:58,131 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-24 00:22:58,367 - INFO - factor=(3, 3), array.shape=(17, 93, 540, 540), vs.shape=(3, 17, 93, 540, 180), hs.shape=(3, 3, 17, 93, 180, 180), untiled.shape=(9, 17, 93, 180, 180)
2026-05-24 00:22:58,369 - INFO - Finished in state Completed()


In [103]:
ecfg = ResizeConfig(width=img.shape[-1], normalize=False, dtype=np.uint16)
labels, l_metadata = Resize(ecfg).run(cropped_labels, metadata=r_metadata)

stackview.blend(
    c_img.astype(np.uint16),
    labels.astype(np.uint64), 
    blend_factor=40 # Sets the transparency of the overlay (0.0 to 1.0)
)

2026-05-24 00:23:06,197 - INFO - Resizing stack from (93, 170, 170) to [93, 512, 512]
2026-05-24 00:23:06,209 - INFO - Running preprocessor Resize, on stack of type uint16, True
2026-05-24 00:23:06,221 - INFO - Running Resize with config: classname='ResizeConfig' package='vistiq.preprocess.preprocess' version='0.0.0.post42+g71fccff.d20260428' command_group=None iterator_config=ArrayIteratorConfig(classname=None, slice_def=(-2, -1)) batch_size=10 preferred_backend='threads' tile_shape=None output_type='stack' output_shape=(93, 512, 512) output_axes=None recompute_scale=True squeeze=True split_axis=None split_channels=False rename_channel=None normalize=False dtype=<class 'numpy.uint16'> width=512 height=None order=1 preserve_range=True anti_aliasing=True
2026-05-24 00:23:06,222 - INFO - StackProcessor.run: received workers=-1 (type: <class 'int'>)
2026-05-24 00:23:06,222 - INFO - Using Parallel with n_jobs=-1 for 93 iterations
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8

# Save label

In [107]:
c_metadata["channel_names"] = "Merged-" + "-".join(c_metadata["channel_names"])

In [109]:
from vistiq.io import ImageWriterConfig, ImageWriter
imc = ImageWriterConfig()
outpath = ".".join(path.split(".")[:-1]) + "-lobes.tif"
ImageWriter(imc).run(labels, outpath, metadata=c_metadata)

2026-05-24 00:30:51,045 - INFO - Preparing to save image with metadata: {'scene_index': 0, 'dim_order': 'ZYX', 'axes': ['Z', 'Y', 'X'], 'channel_names': 'Merged-Scrib-EdU-Dpn', 'channel_axis': 0, 'shape': (93, 512, 512), 'dims': <Dimensions [Z: 93, Y: 512, X: 512]>, 'pixel_unit': 'um', 'scale': Scale(T=None, C=None, Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477), 'physical_pixel_sizes': PhysicalPixelSizes(Z=-0.9999284782608696, Y=0.3000933463796477, X=0.3000933463796477)}
2026-05-24 00:30:51,169 - INFO - Saved image to /standard/vol191/siegristlab/Microsam_Segmentation/24h/AkhGal4 x OR Susie/Scrib488 Dpn555 EdU 647/Raw files/Animal 1-lobes.tif
2026-05-24 00:30:51,171 - INFO - Finished in state Completed()
